# 06 · Chunked prefill with a token budget

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/inference/06_chunked_prefill.ipynb) &nbsp; 📖 Concept deck: [Chunked prefill](https://nilesh-salpe.github.io/visual-ai/inference/04-chunked-prefill/) &nbsp; ⏱️ Suggested time: **40 min**

**You will build:** a Sarathi-style scheduler (2024) that splits long prompts into chunks so prefills never stall ongoing decodes.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def make_requests(n=20, seed=0):
    r = np.random.default_rng(seed); t = 0; out = []
    for i in range(n):
        t += int(r.integers(0, 4))
        out.append({'id': i, 'arrival': t, 'prompt_len': int(r.choice([64, 128, 512, 1500])), 'out_len': int(r.integers(5, 40))})
    return out

def simulate_prefill_first(requests, budget):
    """Baseline: whenever prompts are waiting, run ALL their prefills in one iteration and stall decodes (no budget)."""
    reqs = sorted(requests, key=lambda r: (r['arrival'], r['id'])); i, t = 0, 0
    st = {r['id']: {'left': r['out_len'], 'last': None, 'gaps': []} for r in reqs}; dec, res, tpi = [], {}, []
    while i < len(reqs) or dec:
        waiting = []
        while i < len(reqs) and reqs[i]['arrival'] <= t: waiting.append(reqs[i]); i += 1
        if waiting:
            tpi.append(sum(r['prompt_len'] for r in waiting))
            for r in waiting:
                s = st[r['id']]; s['left'] -= 1; s['last'] = t; res[r['id']] = {'ttft': t + 1 - r['arrival']}
                if s['left'] == 0: res[r['id']]['finish'] = t + 1
                else: dec.append(r)
        elif dec:
            tpi.append(len(dec))
            for r in list(dec):
                s = st[r['id']]; s['gaps'].append(t - s['last']); s['last'] = t; s['left'] -= 1
                if s['left'] == 0: res[r['id']]['finish'] = t + 1; dec.remove(r)
        elif i < len(reqs): t = reqs[i]['arrival']; continue
        t += 1
    return res, tpi
print('helpers ready: make_requests(), simulate_prefill_first()')

## Task 1: Stall-free chunked prefill

Each iteration `t` has a **token budget** `budget`:
1. **Decodes first.** Every request already decoding gets 1 token, so decodes never stall.
2. Fill the remaining budget with **prefill chunks** from requests with `arrival <= t`, in FIFO order (by `arrival`, then `id`). A chunk is `min(prompt tokens left, budget left)`.
3. When a request's prompt finishes in iteration `t`, its **first output token is produced in that same iteration**: `ttft = t + 1 - arrival`. If `out_len == 1` it finishes at `t + 1`; otherwise it starts decoding at `t + 1`.
4. A decoding request finishes at `t + 1` in the iteration that produces its last token.
5. If nothing can run, jump `t` to the next arrival.

Return `(results, tokens_per_iter)`, where `results[id] = {'ttft': …, 'finish': …}` and `tokens_per_iter` lists the tokens used in each iteration that ran. You can assume `budget` is larger than the number of concurrent decodes.

In [ ]:
def simulate_chunked(requests, budget):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _ref_chunked(requests, budget):
    reqs = sorted(requests, key=lambda r: (r['arrival'], r['id']))
    pre = {r['id']: r['prompt_len'] for r in reqs}; left = {r['id']: r['out_len'] for r in reqs}
    res, dec, tpi, t, done = {}, [], [], 0, 0
    while done < len(reqs):
        used = 0
        for r in list(dec):
            used += 1; left[r['id']] -= 1
            if left[r['id']] == 0: res[r['id']]['finish'] = t + 1; dec.remove(r); done += 1
        for r in reqs:
            if r['arrival'] > t or pre[r['id']] == 0 or used >= budget: continue
            c = min(pre[r['id']], budget - used); pre[r['id']] -= c; used += c
            if pre[r['id']] == 0:
                res[r['id']] = {'ttft': t + 1 - r['arrival']}; left[r['id']] -= 1
                if left[r['id']] == 0: res[r['id']]['finish'] = t + 1; done += 1
                else: dec.append(r)
        if used == 0:
            t = min(r['arrival'] for r in reqs if pre[r['id']] > 0); continue
        tpi.append(used); t += 1
    return res, tpi

def _test_simulate_chunked(f):
    for seed, B in [(0, 256), (1, 512), (2, 128)]:
        reqs = make_requests(18, seed)
        res, tpi = f(reqs, B)
        ref, rtpi = _ref_chunked(reqs, B)
        assert max(tpi) <= B, "an iteration exceeded the token budget"
        assert res == ref, f"ttft/finish differ from reference (seed={seed}, budget={B}): decodes first, then FIFO prefill chunks"
        assert tpi == rtpi, "tokens_per_iter differs from reference"

check("simulate_chunked", _test_simulate_chunked, simulate_chunked)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Stall-free chunked prefill
def simulate_chunked_solution(requests, budget):
    reqs = sorted(requests, key=lambda r: (r['arrival'], r['id']))
    prompt_left = {r['id']: r['prompt_len'] for r in reqs}
    out_left = {r['id']: r['out_len'] for r in reqs}
    res, decoding, tokens_per_iter, t, done = {}, [], [], 0, 0
    while done < len(reqs):
        used = 0
        for r in list(decoding):                                   # 1. decodes first: never stall
            used += 1; out_left[r['id']] -= 1
            if out_left[r['id']] == 0:
                res[r['id']]['finish'] = t + 1; decoding.remove(r); done += 1
        for r in reqs:                                             # 2. FIFO prefill chunks in the leftover budget
            if r['arrival'] > t or prompt_left[r['id']] == 0 or used >= budget:
                continue
            chunk = min(prompt_left[r['id']], budget - used)
            prompt_left[r['id']] -= chunk; used += chunk
            if prompt_left[r['id']] == 0:                          # 3. prompt done -> first token now
                res[r['id']] = {'ttft': t + 1 - r['arrival']}; out_left[r['id']] -= 1
                if out_left[r['id']] == 0: res[r['id']]['finish'] = t + 1; done += 1
                else: decoding.append(r)
        if used == 0:                                              # 5. idle -> jump
            t = min(r['arrival'] for r in reqs if prompt_left[r['id']] > 0); continue
        tokens_per_iter.append(used); t += 1
    return res, tokens_per_iter

check("simulate_chunked (reference)", _test_simulate_chunked, simulate_chunked_solution)

---
## Experiments
These use the reference solutions, so they run even before you finish.

In [ ]:
# Iteration cost grows with the tokens it processes. A decode token has to wait for the whole iteration it is in.
reqs = make_requests(20, seed=4)
base, tpi = simulate_prefill_first(reqs, None)
print(f'prefill-first : largest iteration = {max(tpi):5d} tokens -> a decode token can wait behind a {max(tpi)}-token prefill')
for B in (128, 256, 1024):
    res, tpi = simulate_chunked_solution(reqs, B)
    print(f'chunked B={B:4d}: largest iteration = {max(tpi):5d} tokens | iterations: {len(tpi):3d} | mean TTFT {np.mean([v["ttft"] for v in res.values()]):5.1f} iterations')
print('Smaller budgets keep every iteration (and so every decode step) short, but long prompts need more iterations to finish: the TTFT-vs-TPOT knob.')

---
## Takeaways
- Long prefills block decodes in a naive scheduler, so users watching tokens stream see stalls.
- A per-iteration **token budget** with decodes first and prefill chunks after keeps every iteration short and decodes stall-free.
- Sarathi-Serve reported **2.6×** serving capacity for Mistral-7B on one A100, and up to **5.6×** for Falcon-180B, compared with vLLM under latency SLOs.